In [ ]:
!pip install "transformers>=4.40" "datasets<4.0.0" torch torchvision tqdm matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 21.1 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import sys, torch
print(f"Python   : {sys.version.split()[0]}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f"bf16 ok  : {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")


Python   : 3.12.13
PyTorch  : 2.10.0+cu128
CUDA     : 12.8
GPU      : NVIDIA RTX PRO 6000 Blackwell Server Edition
bf16 ok  : True


## Imports

In [ ]:
from __future__ import annotations

import math
import random
import urllib
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from transformers import (
    Dinov2Config,
    Dinov2Model,
)
from transformers.models.dinov2.modeling_dinov2 import Dinov2SelfOutput

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autocast_dtype = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
print(f"Device: {device}  |  autocast dtype: {autocast_dtype}")

# Reproducibility (best-effort)
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Device: cuda  |  autocast dtype: torch.bfloat16


## Config

MAKE USE_GATES True or False depending on what you want to run




In [ ]:
BACKBONE_ID = "facebook/dinov2-small"   # 12 layers, D=384, patch=14
PATCH_SIZE  = 14

# --- Fidelity knobs ---
IMG_SIZE      = 518                     # paper uses 518; must be divisible by 14
BATCH_SIZE    = 2                       # train; eval uses same
EPOCHS        = 10
TRAIN_SUBSET  = 4000                    # None = use all ~47k train images
LR            = 1e-4
WEIGHT_DECAY  = 1e-2

# --- Dataset-specific ---
MIN_DEPTH = 1e-3
MAX_DEPTH = 10.0                        # NYU v2 valid depth range
NUM_BINS  = 256                         # lin.1 / lin.4
NYU_H, NYU_W = 480, 640                 # native NYU resolution for Eigen crop
EIGEN_CROP = (45, 471, 41, 601)         # (top, bottom, left, right) — standard NYU eval crop

# --- Which layers to concat for lin.4 and DPT (12-block backbone → {3,6,9,12}) ---
LIN4_LAYER_INDICES = [3, 6, 9, 12]
DPT_LAYER_INDICES  = [3, 6, 9, 12]

USE_GATES = True

# Sanity
assert IMG_SIZE % PATCH_SIZE == 0, f"IMG_SIZE ({IMG_SIZE}) must be divisible by patch size ({PATCH_SIZE})"
PATCH_GRID = IMG_SIZE // PATCH_SIZE
print(f"Backbone            : {BACKBONE_ID}")
print(f"Input res           : {IMG_SIZE}x{IMG_SIZE}  →  patch grid {PATCH_GRID}x{PATCH_GRID}")
print(f"Train size          : {'FULL (~47k)' if TRAIN_SUBSET is None else f'{TRAIN_SUBSET} images'}")
print(f"Epochs / probe      : {EPOCHS}")
print(f"Batch size          : {BATCH_SIZE}")


Backbone            : facebook/dinov2-small
Input res           : 518x518  →  patch grid 37x37
Train size          : 4000 images
Epochs / probe      : 10
Batch size          : 2


## Gated Self-Attention Module

Verbatim from the original notebook. Drop-in replacement for `Dinov2Attention`
that multiplies the attention output elementwise by a learned sigmoid gate. At
init the gates are ~0.88 (sigmoid(2)), so the model behaves near-identically to
un-gated attention before any training.


In [ ]:
class GatedDinov2SelfAttention(nn.Module):
    """
    DINOv2 multi-head self-attention with an elementwise sigmoid gate
    applied to the SDPA output.

        gate_score = sigmoid(X @ W_gate + b_gate)   # (B, N, H*D)
        attn_out   = softmax(QK^T / sqrt(d)) V
        gated_out  = attn_out * gate_score
    """

    def __init__(self, config: Dinov2Config) -> None:
        super().__init__()

        if config.hidden_size % config.num_attention_heads != 0:
            raise ValueError(
                f"hidden_size ({config.hidden_size}) must be divisible by "
                f"num_attention_heads ({config.num_attention_heads})"
            )

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = config.hidden_size // config.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(config.hidden_size, self.all_head_size, bias=config.qkv_bias)
        self.key   = nn.Linear(config.hidden_size, self.all_head_size, bias=config.qkv_bias)
        self.value = nn.Linear(config.hidden_size, self.all_head_size, bias=config.qkv_bias)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        # Gate projection: hidden_size -> all_head_size
        self.gate_proj = nn.Linear(config.hidden_size, self.all_head_size)
        nn.init.zeros_(self.gate_proj.weight)
        nn.init.constant_(self.gate_proj.bias, 2.0)  # sigmoid(2) ≈ 0.88

    def transpose_for_scores(self, x: torch.Tensor) -> torch.Tensor:
        new_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        return x.view(new_shape).permute(0, 2, 1, 3)

    def forward(
        self,
        hidden_states: torch.Tensor,
        head_mask: Optional[torch.Tensor] = None,
        output_attentions: bool = False,
        **kwargs,
    ) -> Tuple[torch.Tensor, ...]:
        B, N, _ = hidden_states.shape

        gate_score = torch.sigmoid(
            self.transpose_for_scores(self.gate_proj(hidden_states))
        )  # (B, H, N, D)

        Q = self.transpose_for_scores(self.query(hidden_states))
        K = self.transpose_for_scores(self.key(hidden_states))
        V = self.transpose_for_scores(self.value(hidden_states))

        attn_scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.attention_head_size)
        attn_probs  = F.softmax(attn_scores, dim=-1)
        attn_probs  = self.dropout(attn_probs)

        if head_mask is not None:
            attn_probs = attn_probs * head_mask

        context = torch.matmul(attn_probs, V)  # (B, H, N, D)
        context = context * gate_score         # elementwise gate

        context = context.permute(0, 2, 1, 3).contiguous().view(B, N, self.all_head_size)
        outputs = (context, attn_probs) if output_attentions else (context, None)
        return outputs


class GatedDinov2Attention(nn.Module):
    """Drop-in replacement for Dinov2Attention."""

    def __init__(self, config: Dinov2Config) -> None:
        super().__init__()
        self.attention = GatedDinov2SelfAttention(config)
        self.output = Dinov2SelfOutput(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        head_mask: Optional[torch.Tensor] = None,
        output_attentions: bool = False,
        **kwargs,
    ) -> torch.Tensor:
        self_out, attn_weights = self.attention(
            hidden_states, head_mask=head_mask, output_attentions=output_attentions, **kwargs
        )
        return self.output(self_out, hidden_states)


def inject_gated_attention(model: nn.Module, config: Dinov2Config) -> nn.Module:
    encoder = getattr(model, "encoder", None)
    if encoder is None and hasattr(model, "dinov2"):
        encoder = model.dinov2.encoder
    if encoder is None and hasattr(model, "backbone") and hasattr(model.backbone, "encoder"):
        encoder = model.backbone.encoder
    if encoder is None:
        raise ValueError("Could not locate encoder in model.")

    device_now = next(model.parameters()).device
    for layer in encoder.layer:
        layer.attention = GatedDinov2Attention(config).to(device_now)

    n = len(encoder.layer)
    print(f"Injected gated attention into {n} transformer layers.")
    return model


## Load Backbone, Inject Gates, Freeze




In [ ]:
# 1) Load pretrained DINOv2-S (Q/K/V projections have pretrained weights)
backbone = Dinov2Model.from_pretrained(BACKBONE_ID).to(device)
backbone_config = backbone.config

# Stash pretrained attention weights BEFORE injecting gates
pretrained_state = {
    name: param.detach().clone()
    for name, param in backbone.state_dict().items()
    if "attention" in name
}

# 2) Inject gated attention (this replaces Q/K/V with freshly-initialized ones)
if USE_GATES:
  inject_gated_attention(backbone, backbone_config)

# 3) Restore pretrained Q/K/V/bias weights into the new GatedDinov2SelfAttention
missing, unexpected = backbone.load_state_dict(pretrained_state, strict=False)
# Expected: the gate_proj.* params are 'missing' from the pretrained dict -- that's fine.
restored = sum(1 for k in pretrained_state if k not in missing)
print(f"Restored {restored} pretrained attention params.")
gate_keys = [k for k in backbone.state_dict() if "gate_proj" in k]
print(f"Fresh gate params ({len(gate_keys)}): e.g. {gate_keys[:2]}")

# 4) Re-apply gate initialization (W=0, b=2) -- in case load_state_dict perturbed it
def reset_gates(m: nn.Module) -> int:
    n_reset = 0
    for mod in m.modules():
        if isinstance(mod, GatedDinov2SelfAttention):
            nn.init.zeros_(mod.gate_proj.weight)
            nn.init.constant_(mod.gate_proj.bias, 2.0)
            n_reset += 1
    return n_reset

n = reset_gates(backbone)
print(f"Re-initialized gate_proj in {n} attention blocks (W=0, b=2).")

# 5) Freeze everything except gate_proj
for name, param in backbone.named_parameters():
    param.requires_grad = ("gate_proj" in name)

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total     = sum(p.numel() for p in backbone.parameters())
print(f"Backbone trainable params  : {trainable:>10,}  (gates)")
print(f"Backbone frozen params     : {total - trainable:>10,}")
print(f"Backbone total params      : {total:>10,}")

# 6) Force eval mode on backbone -- no dropout / stoch. depth / masking
backbone.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Restored 96 pretrained attention params.
Fresh gate params (0): e.g. []
Re-initialized gate_proj in 0 attention blocks (W=0, b=2).
Backbone trainable params  :          0  (gates)
Backbone frozen params     : 22,056,576
Backbone total params      : 22,056,576


Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-06,

## NYU Depth v2 Dataset (Streamed)




In [ ]:
from datasets import load_dataset
from torchvision import transforms
import torchvision.transforms.functional as TF

_IMG_NORM = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225],
)

print("Opening NYU v2 streaming dataset (no full download)...")
hf_ds = load_dataset("sayakpaul/nyu_depth_v2", streaming=True)
print(hf_ds)


Opening NYU v2 streaming dataset (no full download)...


README.md: 0.00B [00:00, ?B/s]

nyu_depth_v2.py: 0.00B [00:00, ?B/s]

The repository for sayakpaul/nyu_depth_v2 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/sayakpaul/nyu_depth_v2.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
IterableDatasetDict({
    train: IterableDataset({
        features: ['image', 'depth_map'],
        num_shards: 12
    })
    validation: IterableDataset({
        features: ['image', 'depth_map'],
        num_shards: 2
    })
})


In [ ]:
class NYUIterableDataset(torch.utils.data.IterableDataset):
    """
    PyTorch IterableDataset wrapping an HF streaming dataset.

    eval_mode=False: yields (img [3, S, S], depth [1, S, S]) with hflip aug.
    eval_mode=True : yields (img [3, S, S], depth [1, 480, 640]) — native depth.

    Arguments:
      hf_stream      : an HF IterableDataset (a split from streaming=True load).
      take_n         : if set, stops after this many samples per epoch.
      shuffle_buffer : >0 enables a buffered shuffle over the stream.
                       Reseeded each epoch via set_epoch().
    """
    def __init__(self, hf_stream, img_size: int, eval_mode: bool = False,
                 take_n: Optional[int] = None, shuffle_buffer: int = 0,
                 seed: int = 0):
        super().__init__()
        self.hf_stream = hf_stream
        self.img_size = img_size
        self.eval_mode = eval_mode
        self.take_n = take_n
        self.shuffle_buffer = shuffle_buffer
        self.seed = seed
        self._epoch = 0

    def set_epoch(self, epoch: int):
        """Call before each epoch so the shuffle buffer reseeds."""
        self._epoch = epoch

    def _process(self, sample):
        # RGB image
        img = sample["image"].convert("RGB")
        img_t = TF.to_tensor(img)
        img_t = TF.resize(img_t, [self.img_size, self.img_size], antialias=True)

        # Depth map (may be PIL "F" or ndarray, in metres or mm)
        depth_np = np.array(sample["depth_map"], dtype=np.float32)
        if depth_np.max() > 100.0:  # heuristic: mm → m
            depth_np = depth_np / 1000.0
        depth_t = torch.from_numpy(depth_np).unsqueeze(0)

        if self.eval_mode:
            depth_t = TF.resize(
                depth_t, [NYU_H, NYU_W],
                interpolation=TF.InterpolationMode.NEAREST,
            )
        else:
            depth_t = TF.resize(
                depth_t, [self.img_size, self.img_size],
                interpolation=TF.InterpolationMode.NEAREST,
            )
            if random.random() > 0.5:
                img_t   = TF.hflip(img_t)
                depth_t = TF.hflip(depth_t)

        img_t = _IMG_NORM(img_t)
        return img_t, depth_t

    def __iter__(self):
        stream = self.hf_stream
        if self.shuffle_buffer > 0:
            stream = stream.shuffle(seed=self.seed + self._epoch,
                                    buffer_size=self.shuffle_buffer)
        if self.take_n is not None:
            stream = stream.take(self.take_n)

        # Multi-worker sharding: each worker yields a disjoint subset.
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None and worker_info.num_workers > 1:
            stream = stream.shard(num_shards=worker_info.num_workers,
                                  index=worker_info.id)

        for sample in stream:
            yield self._process(sample)


# Known sizes for sayakpaul/nyu_depth_v2 (not available from a streaming split)
NYU_TRAIN_SIZE = 47584
NYU_VAL_SIZE   = 654

train_take = TRAIN_SUBSET if TRAIN_SUBSET is not None else NYU_TRAIN_SIZE

train_ds = NYUIterableDataset(
    hf_ds["train"], img_size=IMG_SIZE, eval_mode=False,
    take_n=train_take,
    shuffle_buffer=1000,   # bigger = better shuffling, more memory
    seed=SEED,
)
val_ds = NYUIterableDataset(
    hf_ds["validation"], img_size=IMG_SIZE, eval_mode=True,
    take_n=None, shuffle_buffer=0, seed=SEED,
)

# DataLoader notes for IterableDataset:
#   - shuffle= must be False (dataset controls order via buffered shuffle)
#   - num_workers=0 is safest; >0 works (we shard inside __iter__)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True,
)

# Streaming datasets have no len() → track expected step counts ourselves.
STEPS_PER_EPOCH_TRAIN = train_take // BATCH_SIZE
STEPS_PER_EPOCH_VAL   = (NYU_VAL_SIZE + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Train samples (streamed): {train_take:,}")
print(f"Val   samples (streamed): {NYU_VAL_SIZE:,}")
print(f"Train steps / epoch     : {STEPS_PER_EPOCH_TRAIN:,}")

# Sanity check — pulls ONE batch from the stream (fresh iterator).
imgs, depths = next(iter(train_loader))
print(f"Train batch: imgs {tuple(imgs.shape)}, depths {tuple(depths.shape)}  "
      f"depth range [{depths.min().item():.3f}, {depths.max().item():.3f}]")
imgs_v, depths_v = next(iter(val_loader))
print(f"Val   batch: imgs {tuple(imgs_v.shape)}, depths {tuple(depths_v.shape)}")
del imgs, depths, imgs_v, depths_v


Train samples (streamed): 4,000
Val   samples (streamed): 654
Train steps / epoch     : 2,000


/usr/local/lib/python3.12/dist-packages/datasets/features/image.py:338: UserWarning: Downcasting array dtype int64 to uint8 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")


Train batch: imgs (2, 3, 518, 518), depths (2, 1, 518, 518)  depth range [0.000, 4.911]
Val   batch: imgs (2, 3, 518, 518), depths (2, 1, 480, 640)


## Probe Heads




In [ ]:
# -------- lin.1 --------
class Lin1Head(nn.Module):
    """Single-layer bin-classification probe. Trainable params: one 1x1 conv."""
    def __init__(self, embed_dim: int, num_bins: int = 256,
                 min_depth: float = MIN_DEPTH, max_depth: float = MAX_DEPTH,
                 upsample: int = 4):
        super().__init__()
        self.num_bins  = num_bins
        self.upsample  = upsample
        self.min_depth = min_depth
        self.max_depth = max_depth
        bin_width = (max_depth - min_depth) / num_bins
        centers = min_depth + bin_width * (torch.arange(num_bins) + 0.5)
        self.register_buffer("bin_centers", centers.view(1, -1, 1, 1))
        self.classifier = nn.Conv2d(2 * embed_dim, num_bins, kernel_size=1)

    def forward(self, hidden_states_list: List[torch.Tensor],
                patch_h: int, patch_w: int, out_size: Tuple[int, int]):
        assert len(hidden_states_list) == 1, "Lin1Head expects 1 layer"
        h = hidden_states_list[0]                                     # [B, 1+N, D]
        B, T, D = h.shape
        assert T == 1 + patch_h * patch_w, f"Expected {1 + patch_h*patch_w} tokens, got {T}"
        cls, patches = h[:, 0], h[:, 1:]                              # [B,D], [B,N,D]
        patches = patches.transpose(1, 2).reshape(B, D, patch_h, patch_w)
        cls_bcast = cls[:, :, None, None].expand(-1, -1, patch_h, patch_w)
        feats = torch.cat([patches, cls_bcast], dim=1)                # [B, 2D, h, w]
        feats = F.interpolate(feats, scale_factor=self.upsample,
                              mode="bilinear", align_corners=False)   # [B, 2D, 4h, 4w]
        logits = self.classifier(feats)                               # [B, K, 4h, 4w]
        probs  = F.softmax(logits, dim=1)
        depth  = (probs * self.bin_centers).sum(dim=1, keepdim=True)  # [B, 1, 4h, 4w]
        depth  = F.interpolate(depth, size=out_size, mode="bilinear", align_corners=False)
        return {"depth": depth, "logits": logits}


# -------- lin.4 --------
class Lin4Head(nn.Module):
    """Four-layer bin-classification probe: [B, 5D, h, w] → 256 bins."""
    def __init__(self, embed_dim: int, num_layers: int = 4,
                 num_bins: int = 256, min_depth: float = MIN_DEPTH,
                 max_depth: float = MAX_DEPTH, upsample: int = 4):
        super().__init__()
        self.num_bins  = num_bins
        self.upsample  = upsample
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.num_layers = num_layers
        bin_width = (max_depth - min_depth) / num_bins
        centers = min_depth + bin_width * (torch.arange(num_bins) + 0.5)
        self.register_buffer("bin_centers", centers.view(1, -1, 1, 1))
        # Input channels: num_layers * D  (patches)  +  D  (CLS broadcast)
        in_ch = (num_layers + 1) * embed_dim
        self.classifier = nn.Conv2d(in_ch, num_bins, kernel_size=1)

    def forward(self, hidden_states_list: List[torch.Tensor],
                patch_h: int, patch_w: int, out_size: Tuple[int, int]):
        assert len(hidden_states_list) == self.num_layers
        feats_list = []
        for h in hidden_states_list:
            B, T, D = h.shape
            patches = h[:, 1:].transpose(1, 2).reshape(B, D, patch_h, patch_w)
            feats_list.append(patches)
        # Broadcast CLS from the deepest layer
        cls = hidden_states_list[-1][:, 0]                             # [B, D]
        feats_list.append(cls[:, :, None, None].expand(-1, -1, patch_h, patch_w))
        feats = torch.cat(feats_list, dim=1)                           # [B, 5D, h, w]
        feats = F.interpolate(feats, scale_factor=self.upsample,
                              mode="bilinear", align_corners=False)
        logits = self.classifier(feats)
        probs  = F.softmax(logits, dim=1)
        depth  = (probs * self.bin_centers).sum(dim=1, keepdim=True)
        depth  = F.interpolate(depth, size=out_size, mode="bilinear", align_corners=False)
        return {"depth": depth, "logits": logits}


In [ ]:
# -------- DPT (compact, Ranftl et al. 2021 style) --------

class DPTReassemble(nn.Module):
    """Readout-project + 1x1 proj + resample to target scale of the patch grid."""
    def __init__(self, embed_dim: int, out_channels: int, scale: float):
        super().__init__()
        # "project" readout: concat CLS to every patch, then project back to D
        self.readout_proj = nn.Sequential(
            nn.Linear(2 * embed_dim, embed_dim),
            nn.GELU(),
        )
        self.proj = nn.Conv2d(embed_dim, out_channels, kernel_size=1)
        self.scale = scale
        if scale > 1.0:
            self.resample = nn.ConvTranspose2d(out_channels, out_channels,
                                               kernel_size=int(scale), stride=int(scale))
        elif scale < 1.0:
            k = int(round(1.0 / scale))
            self.resample = nn.Conv2d(out_channels, out_channels, kernel_size=k, stride=k)
        else:
            self.resample = nn.Identity()

    def forward(self, hidden_state: torch.Tensor, patch_h: int, patch_w: int):
        B, T, D = hidden_state.shape
        cls     = hidden_state[:, :1]                                  # [B, 1, D]
        patches = hidden_state[:, 1:]                                  # [B, N, D]
        cls_bc  = cls.expand(-1, patches.shape[1], -1)                 # [B, N, D]
        x = torch.cat([patches, cls_bc], dim=-1)                       # [B, N, 2D]
        x = self.readout_proj(x)                                       # [B, N, D]
        x = x.transpose(1, 2).reshape(B, D, patch_h, patch_w)          # [B, D, h, w]
        x = self.proj(x)                                               # [B, C, h, w]
        x = self.resample(x)                                           # [B, C, h', w']
        return x


class FusionBlock(nn.Module):
    """RefineNet-style fusion: residual refine + skip add + bilinear upsample to skip size."""
    def __init__(self, channels: int):
        super().__init__()
        self.res1 = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.ReLU(inplace=False),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
        )
        self.res2 = nn.Sequential(
            nn.ReLU(inplace=False),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.ReLU(inplace=False),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
        )

    def forward(self, x: torch.Tensor, skip: Optional[torch.Tensor] = None) -> torch.Tensor:
        if skip is not None:
            skip = skip + self.res1(skip)
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = x + skip
        x = x + self.res2(x)
        return x


class DPTHead(nn.Module):
    """Four-layer DPT decoder with SILog-friendly depth output in [min_depth, max_depth]."""
    # Scales are relative to the patch grid (1.0 = same as grid).
    # Shallowest layer → largest map; deepest → smallest.
    SCALES = [4.0, 2.0, 1.0, 0.5]

    def __init__(self, embed_dim: int, hidden_dim: int = 64,
                 min_depth: float = MIN_DEPTH, max_depth: float = MAX_DEPTH):
        super().__init__()
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.reassemble = nn.ModuleList([
            DPTReassemble(embed_dim, hidden_dim, s) for s in self.SCALES
        ])
        self.fusion = nn.ModuleList([FusionBlock(hidden_dim) for _ in range(4)])
        self.depth_head = nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1),
            nn.ReLU(inplace=False),
            nn.Conv2d(hidden_dim, 32, 3, padding=1),
            nn.ReLU(inplace=False),
            nn.Conv2d(32, 1, 1),
        )

    def forward(self, hidden_states_list: List[torch.Tensor],
                patch_h: int, patch_w: int, out_size: Tuple[int, int]):
        assert len(hidden_states_list) == 4, "DPTHead expects 4 layers (shallow→deep)"
        pyramid = [
            self.reassemble[i](hidden_states_list[i], patch_h, patch_w)
            for i in range(4)
        ]
        # Top-down fusion (deepest first, no skip)
        x = self.fusion[3](pyramid[3])
        x = self.fusion[2](x, pyramid[2])
        x = self.fusion[1](x, pyramid[1])
        x = self.fusion[0](x, pyramid[0])
        raw   = self.depth_head(x)
        # Bounded depth: sigmoid → [min, max]
        depth = torch.sigmoid(raw) * (self.max_depth - self.min_depth) + self.min_depth
        depth = F.interpolate(depth, size=out_size, mode="bilinear", align_corners=False)
        return {"depth": depth, "logits": None}


## Losses, Metrics, Eigen Crop



In [ ]:
def bin_cls_loss(logits: torch.Tensor, target_depth: torch.Tensor,
                 min_depth: float = MIN_DEPTH, max_depth: float = MAX_DEPTH) -> torch.Tensor:
    """
    logits:       [B, K, h, w] predictions
    target_depth: [B, 1, H, W] ground truth (any resolution, will be nearest-down-sampled)
    """
    B, K, h, w = logits.shape
    t = F.interpolate(target_depth, size=(h, w), mode="nearest")       # [B, 1, h, w]
    bin_width = (max_depth - min_depth) / K
    idx = ((t - min_depth) / bin_width).long().clamp_(0, K - 1).squeeze(1)  # [B, h, w]
    valid = ((t > min_depth) & (t < max_depth) & torch.isfinite(t)).squeeze(1)
    if valid.sum() == 0:
        return logits.sum() * 0.0
    logits_f = logits.permute(0, 2, 3, 1).reshape(-1, K)
    idx_f    = idx.reshape(-1)
    valid_f  = valid.reshape(-1)
    return F.cross_entropy(logits_f[valid_f], idx_f[valid_f])


def silog_loss(pred: torch.Tensor, target: torch.Tensor,
               min_depth: float = MIN_DEPTH, max_depth: float = MAX_DEPTH,
               lam: float = 0.85, alpha: float = 10.0) -> torch.Tensor:
    """Eigen et al. 2014 scale-invariant log loss, masked + scaled."""
    valid = (target > min_depth) & (target < max_depth) & torch.isfinite(target)
    if valid.sum() == 0:
        return pred.sum() * 0.0
    p = pred[valid].clamp(min=1e-6)
    t = target[valid].clamp(min=1e-6)
    g = torch.log(p) - torch.log(t)
    var = (g ** 2).mean() - lam * (g.mean() ** 2)
    return alpha * torch.sqrt(var.clamp(min=1e-8))


@torch.no_grad()
def compute_metrics_nyu(pred: torch.Tensor, target_native: torch.Tensor,
                        eigen_crop: Tuple[int, int, int, int] = EIGEN_CROP,
                        min_depth: float = MIN_DEPTH,
                        max_depth: float = MAX_DEPTH) -> Dict[str, float]:
    """
    pred         : [B, 1, H_pred, W_pred] — will be upsampled to NYU_H x NYU_W
    target_native: [B, 1, NYU_H, NYU_W]    — kept at native res
    Applies Eigen crop before computing RMSE / AbsRel / delta<1.25.
    """
    pred_up = F.interpolate(pred.float(), size=(NYU_H, NYU_W), mode="bilinear", align_corners=False)
    t, b, l, r = eigen_crop
    p = pred_up[:, :, t:b, l:r].clamp(min_depth, max_depth)
    y = target_native[:, :, t:b, l:r]
    valid = (y > min_depth) & (y < max_depth) & torch.isfinite(y)
    if valid.sum() == 0:
        return {"rmse": float("nan"), "abs_rel": float("nan"), "delta1": float("nan"), "n": 0}
    p = p[valid]
    y = y[valid]
    rmse    = torch.sqrt(((p - y) ** 2).mean()).item()
    abs_rel = ((p - y).abs() / y).mean().item()
    ratio   = torch.maximum(p / y, y / p)
    delta1  = (ratio < 1.25).float().mean().item()
    return {"rmse": rmse, "abs_rel": abs_rel, "delta1": delta1, "n": int(valid.sum().item())}


## Train




In [ ]:
from tqdm.auto import tqdm

def get_backbone_features(backbone: nn.Module, imgs: torch.Tensor,
                          layer_indices: List[int]) -> Tuple[List[torch.Tensor], int, int]:
    """
    Returns: (list of hidden states at `layer_indices` [B, 1+N, D], patch_h, patch_w)
    layer_indices are 1-indexed transformer layer outputs (1..num_layers).
    """
    # backbone.eval() is set elsewhere; dropout / stoch. depth are disabled.
    out = backbone(
        pixel_values=imgs,
        output_hidden_states=True,
        interpolate_pos_encoding=True,
    )
    # HF: hidden_states is a tuple of length (num_layers + 1); [0] is embedding, [i] is layer-i output.
    hs = out.hidden_states
    return [hs[i] for i in layer_indices], imgs.shape[-2] // PATCH_SIZE, imgs.shape[-1] // PATCH_SIZE


@torch.no_grad()
def evaluate(backbone: nn.Module, head: nn.Module, loader: DataLoader,
             layer_indices: List[int]) -> Dict[str, float]:
    head.eval()
    totals = {"rmse": 0.0, "abs_rel": 0.0, "delta1": 0.0, "n": 0}
    num_batches = 0
    for imgs, depths_native in loader:
        imgs = imgs.to(device, non_blocking=True)
        depths_native = depths_native.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=autocast_dtype, enabled=(autocast_dtype != torch.float32)):
            hs_list, ph, pw = get_backbone_features(backbone, imgs, layer_indices)
            # For eval we want depth at NYU native res so metrics crop is exact.
            out = head(hs_list, ph, pw, out_size=(NYU_H, NYU_W))
        depth_pred = out["depth"].float()
        m = compute_metrics_nyu(depth_pred, depths_native)
        for k in ("rmse", "abs_rel", "delta1"):
            totals[k] += m[k]
        totals["n"] += m["n"]
        num_batches += 1
    return {k: totals[k] / max(num_batches, 1) for k in ("rmse", "abs_rel", "delta1")}


def train_probe(
    backbone: nn.Module,
    head: nn.Module,
    probe_name: str,
    loss_type: str,                    # "bin_cls" | "silog"
    layer_indices: List[int],
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    steps_per_epoch: int = STEPS_PER_EPOCH_TRAIN,
) -> Dict[str, float]:
    """Trains `head` and the backbone gates. Returns best validation metrics."""
    head = head.to(device)
    assert loss_type in ("bin_cls", "silog")

    # 1. Reset gates for a clean start
    reset_gates(backbone)

    # 2. Build optimizer over (gate params ∪ head params)
    gate_params = [p for n, p in backbone.named_parameters() if "gate_proj" in n and p.requires_grad]
    head_params = list(head.parameters())
    n_gate = sum(p.numel() for p in gate_params)
    n_head = sum(p.numel() for p in head_params)
    optimizer = torch.optim.AdamW(
        [{"params": gate_params, "lr": lr}, {"params": head_params, "lr": lr}],
        betas=(0.9, 0.999), weight_decay=weight_decay,
    )
    total_steps = max(epochs * steps_per_epoch, 1)
    scheduler = torch.optim.lr_scheduler.PolynomialLR(
        optimizer, total_iters=total_steps, power=0.9
    )

    print(f"\n=== Training {probe_name} ===")
    print(f"  gate params: {n_gate:>8,}  |  head params: {n_head:>8,}  |  total trainable: {n_gate+n_head:>8,}")
    print(f"  layers used: {layer_indices}  |  loss: {loss_type}  |  epochs: {epochs}  |  lr: {lr}")
    print(f"  steps/epoch: {steps_per_epoch}  |  total steps: {total_steps}")

    best = {"rmse": float("inf"), "abs_rel": float("inf"), "delta1": 0.0}

    for epoch in range(1, epochs + 1):
        # Reseed the streaming shuffle buffer so epochs see different orderings
        if hasattr(train_loader.dataset, "set_epoch"):
            train_loader.dataset.set_epoch(epoch)

        backbone.eval()   # backbone stays in eval, but gate params still receive gradients
        head.train()
        running = 0.0
        num_batches = 0
        pbar = tqdm(train_loader, total=steps_per_epoch,
                    desc=f"{probe_name} Ep {epoch}/{epochs}", leave=False)
        for imgs, depths in pbar:
            imgs   = imgs.to(device, non_blocking=True)
            depths = depths.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=autocast_dtype, enabled=(autocast_dtype != torch.float32)):
                hs_list, ph, pw = get_backbone_features(backbone, imgs, layer_indices)
                out = head(hs_list, ph, pw, out_size=depths.shape[-2:])
                if loss_type == "bin_cls":
                    loss = bin_cls_loss(out["logits"].float(), depths.float())
                else:
                    loss = silog_loss(out["depth"].float(), depths.float())

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gate_params + head_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running += loss.item()
            num_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = running / max(num_batches, 1)
        metrics  = evaluate(backbone, head, val_loader, layer_indices)
        print(f"  Ep {epoch:>2}/{epochs}  train_loss={avg_loss:.4f}  "
              f"rmse={metrics['rmse']:.4f}  abs_rel={metrics['abs_rel']:.4f}  delta1={metrics['delta1']:.4f}")
        if metrics["rmse"] < best["rmse"]:
            best = dict(metrics)

    print(f"  >> Best {probe_name}: rmse={best['rmse']:.4f}  abs_rel={best['abs_rel']:.4f}  delta1={best['delta1']:.4f}")
    return best


## Run — lin.1 probe

In [ ]:
# D = backbone_config.hidden_size
# head_lin1 = Lin1Head(embed_dim=D, num_bins=NUM_BINS,
#                      min_depth=MIN_DEPTH, max_depth=MAX_DEPTH)
# results_lin1 = train_probe(
#     backbone, head_lin1,
#     probe_name="lin.1",
#     loss_type="bin_cls",
#     layer_indices=[12],                # last layer only
#     train_loader=train_loader,
#     val_loader=val_loader,
# )
# torch.save({"head": head_lin1.state_dict(),
#             "gates": {n: p.detach().cpu() for n, p in backbone.named_parameters() if "gate_proj" in n},
#             "metrics": results_lin1}, "probe_lin1.pt")


## Run — lin.4 probe

In [ ]:
# D = backbone_config.hidden_size
# head_lin4 = Lin4Head(embed_dim=D, num_layers=len(LIN4_LAYER_INDICES),
#                      num_bins=NUM_BINS, min_depth=MIN_DEPTH, max_depth=MAX_DEPTH)
# results_lin4 = train_probe(
#     backbone, head_lin4,
#     probe_name="lin.4",
#     loss_type="bin_cls",
#     layer_indices=LIN4_LAYER_INDICES,
#     train_loader=train_loader,
#     val_loader=val_loader,
# )
# torch.save({"head": head_lin4.state_dict(),
#             "gates": {n: p.detach().cpu() for n, p in backbone.named_parameters() if "gate_proj" in n},
#             "metrics": results_lin4}, "probe_lin4.pt")


## Run — DPT probe

In [ ]:
D = backbone_config.hidden_size
head_dpt = DPTHead(embed_dim=D, hidden_dim=64,
                   min_depth=MIN_DEPTH, max_depth=MAX_DEPTH)
results_dpt = train_probe(
    backbone, head_dpt,
    probe_name="DPT",
    loss_type="silog",
    layer_indices=DPT_LAYER_INDICES,
    train_loader=train_loader,
    val_loader=val_loader,
)
tag = "gated" if USE_GATES else "ungated"
torch.save({"head": head_dpt.state_dict(),
            "gates": {n: p.detach().cpu() for n, p in backbone.named_parameters() if "gate_proj" in n},
            "metrics": results_dpt}, f"probe_dpt_{tag}.pt")



=== Training DPT ===
  gate params:        0  |  head params: 2,023,489  |  total trainable: 2,023,489
  layers used: [3, 6, 9, 12]  |  loss: silog  |  epochs: 10  |  lr: 0.0001
  steps/epoch: 2000  |  total steps: 20000


DPT Ep 1/10:   0%|          | 0/2000 [00:00<?, ?it/s]

  Ep  1/10  train_loss=1.4604  rmse=0.5667  abs_rel=0.1701  delta1=0.7457


DPT Ep 2/10:   0%|          | 0/2000 [00:00<?, ?it/s]

  Ep  2/10  train_loss=1.1100  rmse=0.5455  abs_rel=0.1439  delta1=0.7901


...

  Ep  8/10  train_loss=1.0960  rmse=0.4609  abs_rel=0.1253  delta1=0.8496


DPT Ep 9/10:   0%|          | 0/2000 [00:00<?, ?it/s]

  Ep  9/10  train_loss=0.9769  rmse=0.4653  abs_rel=0.1278  delta1=0.8467


DPT Ep 10/10:   0%|          | 0/2000 [00:00<?, ?it/s]

  Ep 10/10  train_loss=0.7733  rmse=0.4725  abs_rel=0.1278  delta1=0.8467
  >> Best DPT: rmse=0.4609  abs_rel=0.1253  delta1=0.8496


## Summary 


In [ ]:
print(f"{'Probe':<8} {'Trainable':>12} {'RMSE':>8} {'AbsRel':>8} {'δ<1.25':>8}")
print("-" * 50)
def _count_params(head):
    return sum(p.numel() for p in head.parameters())
rows = [
    # ("lin.1", _count_params(head_lin1), results_lin1),
    # ("lin.4", _count_params(head_lin4), results_lin4),
    ("DPT",   _count_params(head_dpt),  results_dpt),
]
for name, n_params, m in rows:
    print(f"{name:<8} {n_params:>12,} {m['rmse']:>8.4f} {m['abs_rel']:>8.4f} {m['delta1']:>8.4f}")

print(f"\nConfig: IMG_SIZE={IMG_SIZE}, EPOCHS={EPOCHS}, TRAIN_SUBSET={TRAIN_SUBSET}, BS={BATCH_SIZE}")


Probe       Trainable     RMSE   AbsRel   δ<1.25
--------------------------------------------------
DPT         2,023,489   0.4609   0.1253   0.8496

Config: IMG_SIZE=518, EPOCHS=10, TRAIN_SUBSET=4000, BS=2
